# 36 — Evaluación final TEST y capacidad de revisión

**Este es el único notebook de M4 autorizado a consumir TEST.** La selección de modelo y calibrador ya debe estar congelada. La primera ejecución materializa la evaluación final; ejecuciones posteriores reutilizan los artefactos completos y no vuelven a evaluar TEST.

In [1]:
from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec
from argos.config.modeling import load_modeling_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

m3_gate = paths.docs / "decisions" / "M3_GATE.md"
assert m3_gate.exists() and "M3 ✅ CLOSED" in m3_gate.read_text(encoding="utf-8"), (
    "M3 no está formalmente cerrado. No iniciar Modeling."
)
GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")
MODEL_SPEC = load_modeling_spec(paths.configs / "modeling.toml")
MODEL_DIR = paths.root / "artifacts" / "models" / "m4"
MANIFEST_DIR = paths.root / "artifacts" / "manifests"

{
    "gold_schema": GOLD_SPEC.schema_version,
    "modeling_schema": MODEL_SPEC.schema_version,
    "target": MODEL_SPEC.target,
    "model_matrix": MODEL_SPEC.model_matrix,
    "operational_review_fraction": MODEL_SPEC.ranking.operational_fraction,
}

assert (paths.reports / "modeling" / "35_validation.parquet").exists(), "Ejecuta 35 primero."
from argos.models.business import BUILD_IMPLEMENTATION
BUILD_IMPLEMENTATION


'm4-36-final-test-business-impact-v1'

In [2]:
from argos.models.business import evaluate_final_test

result = evaluate_final_test(
    root=paths.root,
    gold_spec=GOLD_SPEC,
    modeling_spec=MODEL_SPEC,
    reports_root=paths.reports,
    model_dir=MODEL_DIR,
    manifest_dir=MANIFEST_DIR,
    docs_root=paths.docs,
)
{"reused_existing_test_artifacts": result.reused, "model_card": str(result.model_card_path)}


{'reused_existing_test_artifacts': False,
 'model_card': '/Users/macm1pro/Desktop/projects/argos/argos-code/docs/models/ARGOS_V1_MODEL_CARD.md'}

In [3]:
result.metrics

,model,split,rows,positives,prevalence,pr_auc,roc_auc,log_loss,brier,ece
0,hist_gradient_boosting+isotonic,test,1050857,10983,0.010451,0.13525,0.872745,0.045136,0.009638,0.002843


In [4]:
result.ranking

,model,split,review_fraction,reviewed,captured_positives,precision_at_k,recall_at_k,lift_at_k,cumulative_gain
0,hist_gradient_boosting+isotonic,test,0.001,1051,283,0.269267,0.025767,25.763589,0.025767
1,hist_gradient_boosting+isotonic,test,0.005,5255,1388,0.264129,0.126377,25.271987,0.126377
2,hist_gradient_boosting+isotonic,test,0.010,10509,2776,0.264155,0.252754,25.274391,0.252754
3,hist_gradient_boosting+isotonic,test,0.025,26272,4293,0.163406,0.390877,15.634730,0.390877
4,hist_gradient_boosting+isotonic,test,0.050,52543,5759,0.109605,0.524356,10.487087,0.524356
5,hist_gradient_boosting+isotonic,test,0.100,105086,7143,0.067973,0.650369,6.503669,0.650369


In [5]:
result.reliability

,model,split,bin,rows,mean_predicted_probability,observed_rate,min_score,max_score
0,hist_gradient_boosting+isotonic,test,0,322079,0.002692,0.000935,1.000000e-07,0.002712
1,hist_gradient_boosting+isotonic,test,1,55094,0.002738,0.001379,2.712763e-03,0.002738
2,hist_gradient_boosting+isotonic,test,2,55477,0.002831,0.001803,2.767485e-03,0.002831
3,hist_gradient_boosting+isotonic,test,3,88506,0.002936,0.002463,2.886473e-03,0.002936
4,hist_gradient_boosting+isotonic,test,4,82383,0.003574,0.003739,3.133734e-03,0.003658
5,hist_gradient_boosting+isotonic,test,5,86814,0.004820,0.004377,3.878897e-03,0.004873
6,hist_gradient_boosting+isotonic,test,6,23941,0.005429,0.004052,5.333333e-03,0.005431
7,hist_gradient_boosting+isotonic,test,7,75514,0.005901,0.006502,5.434318e-03,0.006448
8,hist_gradient_boosting+isotonic,test,8,57445,0.008766,0.008565,7.004532e-03,0.011525
9,hist_gradient_boosting+isotonic,test,9,77282,0.017637,0.012771,1.244959e-02,0.019697


In [6]:
result.validation

,check_id,category,blocking,passed,observed,requirement
0,final_test_population,evaluation,True,True,"{""distinct_contracts"": 1050857, ""expected"": 10...",final evaluation covers the complete frozen TE...
1,final_test_probabilities_finite,quality,True,True,"{""max"": 0.130938450737187, ""min"": 1e-07}",final TEST probabilities are finite and bounded
2,selection_artifacts_unchanged,test_isolation,True,True,"{""calibrator_sha256"": ""95e8e506f92b191f7315a1f...",model and calibrator match the hashes frozen b...
3,final_test_metrics_finite,evaluation,True,True,"{""brier"": 0.009637963487861365, ""ece"": 0.00284...",all predeclared final TEST probability metrics...
4,final_test_operational_lift,usefulness,True,True,"{""lift"": 25.274391455433765, ""minimum"": 1.0, ""...",frozen model provides lift above random review...


In [7]:
blocking = result.validation.loc[result.validation["blocking"]]
print("36 ✅ PASS" if result.ready_for_gate else "36 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Final TEST manifest:", result.manifest_path)
assert result.ready_for_gate


36 ✅ PASS
Blocking checks: 5/5
Final TEST manifest: /Users/macm1pro/Desktop/projects/argos/argos-code/artifacts/manifests/m4_final_test.json
